# Recall@5 per chunker on COVID-QA (BM25 fixed)

`02-covidqa-intrinsic-metrics.ipynb` measures chunks before retrieval: whether an
answer stays whole inside one chunk, whether tables keep their headers, and whether
chunks carry a section path. This notebook asks the retrieval question those metrics
point at: **when the chunker changes, does the right chunk come back?**

Setup:

- 137 COVID-QA papers, 1,852 expert questions (`gold/questions.jsonl`)
- one retriever for every row: BM25, so the chunker is the only thing that changes
- a hit = one of the top 5 chunks is from the question's paper **and** contains the
  normalised gold answer text

Three chunkers, all with a 512-token budget and the same tokenizer:

| Row | Chunker | What BM25 indexes |
|---|---|---|
| `fixed_512` | `RecursiveCharacterTextSplitter` over the paper's Markdown, 50-token overlap | the chunk text |
| `docling_512` | Docling `HybridChunker` | the chunk text |
| `docling_512_label` | Docling `HybridChunker` | `Paper: <title>` + `contextualize()` (heading path) + chunk text |

The third row is the cheap form of contextual retrieval: each chunk carries the
title of its paper and its section headings when it is indexed.

## Capabilities

| Name | What it does | Example |
|---|---|---|
| `norm` | lowercases and strips punctuation for text matching | `"Fourteen days."` → `"fourteen days"` |
| `chunk_fixed` | fixed 512-token chunks of one paper | ~36 chunks per paper |
| `chunk_docling` | Docling chunks, plain and labelled | ~43 chunks per paper |
| `recall_at_5` | BM25 top-5 hit list for one chunk set | `[1, 0, 1, ...]` |
| `answer_findable` | share of answers that sit whole inside some chunk | the ceiling for any retriever |
| `paired_ci` | paired bootstrap 95% CI of the difference between two rows | `(+0.004, -0.003, +0.011)` |

Runs offline once the data is on disk (no API key, no GPU). Chunking all 137 papers
takes about 2-4 minutes on one CPU.

### Step 1 · Bootstrap the environment

In [1]:
import sys
from pathlib import Path

_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()
REPO = _root
print("repository root found")

repository root found


### Step 2 · Locate the data

The benchmark lives in `04-benchmarks/covid-qa/data/` when `build_gold.py` has been run
locally. Otherwise the parsed papers and gold questions are downloaded from the
`anacodicai-labs/covidqa-bench` dataset on Hugging Face (about 1.7 GB, once).

In [2]:
import json
import tarfile

DATA = REPO / "04-benchmarks" / "covid-qa" / "data"
PARSED = DATA / "02_parsed" / "covidqa"
GOLD = DATA / "gold" / "questions.jsonl"


def ensure_data():
    if GOLD.is_file() and PARSED.is_dir():
        return
    import zstandard as zstd
    from huggingface_hub import hf_hub_download
    DATA.mkdir(parents=True, exist_ok=True)
    hf_hub_download("anacodicai-labs/covidqa-bench", "gold/questions.jsonl",
                    repo_type="dataset", local_dir=str(DATA))
    archive = hf_hub_download("anacodicai-labs/covidqa-bench", "parsed_questions_137.tar.zst",
                              repo_type="dataset", local_dir=str(DATA))
    with open(archive, "rb") as fh:
        reader = zstd.ZstdDecompressor().stream_reader(fh)
        with tarfile.open(fileobj=reader, mode="r|") as tar:
            tar.extractall(DATA)

### Step 3 · Load the gold questions

In [3]:
ensure_data()
questions = []
for line in open(GOLD, encoding="utf-8"):
    if line.strip():
        questions.append(json.loads(line))

paper_ids = sorted({q["paper_id"] for q in questions})
print(len(questions), "questions over", len(paper_ids), "papers")
print("example:", questions[0]["question"], "->", questions[0]["gold_answer"])

1852 questions over 137 papers
example: What health regulations were changes due to the outbreak of C. burnetti? -> forbid the display of sheep in the latter third of their pregnancy and require regular testing of animals for C. burnetii in petting zoos


### Step 4 · Define the text normaliser

The same normaliser is used for the answer check and for BM25 tokens, so a match never
fails on case or punctuation alone.

In [4]:
import re
import unicodedata


def norm(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"-\s*\n\s*", "", s)
    return re.sub(r"[^a-z0-9]+", " ", s).strip()


print(norm("Fourteen days."), "|", norm("SARS-CoV-2 (COVID-19)"))

fourteen days | sars cov 2 covid 19


### Step 5 · Build the two chunkers

Both use Docling's tokenizer, so "512 tokens" means the same thing in every row.

In [5]:
from docling.chunking import HybridChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter

docling_chunker = HybridChunker(max_tokens=512)
tokenizer = docling_chunker.tokenizer
fixed_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
    length_function=tokenizer.count_tokens,
)
print("chunkers ready")

chunkers ready


### Step 6 · Define the per-paper chunk functions

`text` is what the answer check reads. `index_text` is what BM25 indexes. Only the
labelled row makes them differ.

In [6]:
from docling_core.types.doc import DoclingDocument


def paper_title(doc):
    for item in doc.texts:
        label = getattr(item.label, "value", str(item.label))
        if label == "title" and item.text.strip():
            return item.text.strip()
    return doc.name


def chunk_fixed(doc, pid):
    rows = []
    for text in fixed_splitter.split_text(doc.export_to_markdown()):
        rows.append({"paper": pid, "text": text, "index_text": text})
    return rows


def chunk_docling(doc, pid):
    plain = []
    labelled = []
    title = paper_title(doc)
    for ch in docling_chunker.chunk(doc):
        plain.append({"paper": pid, "text": ch.text, "index_text": ch.text})
        labelled.append({"paper": pid, "text": ch.text,
                         "index_text": "Paper: " + title + "\n" + docling_chunker.contextualize(ch)})
    return plain, labelled

### Step 7 · Look at one paper before running all of them

In [7]:
sample_id = paper_ids[0]
sample_doc = DoclingDocument.load_from_json(PARSED / sample_id / (sample_id + ".docling.json"))
sample_fixed = chunk_fixed(sample_doc, sample_id)
sample_plain, sample_labelled = chunk_docling(sample_doc, sample_id)
print(sample_id, "| fixed:", len(sample_fixed), "chunks | docling:", len(sample_plain), "chunks")
print("\n--- labelled chunk, as BM25 sees it ---")
print(sample_labelled[3]["index_text"][:400])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (581 > 512). Running this sequence through the model will result in indexing errors


pmc1618839 | fixed: 31 chunks | docling: 49 chunks

--- labelled chunk, as BM25 sees it ---
Paper: PMC1618839
Abstract
Background: In May 2003 the Soest County Health Department was informed of an unusually large number of patients hospitalized with atypical pneumonia.
Methods: In exploratory interviews patients mentioned having visited a farmers' market where a sheep had lambed. Serologic testing confirmed the diagnosis of Q fever. We asked local health departments in Germany to identiy


### Step 8 · Chunk all 137 papers three ways

In [8]:
import time

chunk_sets = {"fixed_512": [], "docling_512": [], "docling_512_label": []}
t0 = time.time()
for pid in paper_ids:
    doc = DoclingDocument.load_from_json(PARSED / pid / (pid + ".docling.json"))
    chunk_sets["fixed_512"].extend(chunk_fixed(doc, pid))
    plain, labelled = chunk_docling(doc, pid)
    chunk_sets["docling_512"].extend(plain)
    chunk_sets["docling_512_label"].extend(labelled)

print(f"chunked {len(paper_ids)} papers in {time.time() - t0:.0f}s")
for name in chunk_sets:
    print(f"  {name:<18} {len(chunk_sets[name]):>6} chunks")

chunked 137 papers in 120s
  fixed_512            4970 chunks
  docling_512          5912 chunks
  docling_512_label    5912 chunks


### Step 9 · Define the ceiling: is the answer whole inside some chunk?

If a chunk boundary cuts the answer in two, no retriever can return it as one hit.
This share is the best Recall any retriever could reach on that chunk set.

In [9]:
def answer_findable(chunks):
    by_paper = {}
    for c in chunks:
        by_paper.setdefault(c["paper"], []).append(norm(c["text"]))
    found = 0
    for q in questions:
        ans = norm(q["gold_answer"])
        for t in by_paper.get(q["paper_id"], []):
            if ans and ans in t:
                found += 1
                break
    return found / len(questions)

### Step 10 · Define Recall@5 with BM25

In [10]:
from rank_bm25 import BM25Okapi


def recall_at_5(chunks):
    corpus = []
    texts = []
    for c in chunks:
        corpus.append(norm(c["index_text"]).split())
        texts.append(norm(c["text"]))
    bm25 = BM25Okapi(corpus)
    hits = []
    for q in questions:
        scores = bm25.get_scores(norm(q["question"]).split())
        top = sorted(range(len(scores)), key=lambda j: -scores[j])[:5]
        ans = norm(q["gold_answer"])
        hit = 0
        for j in top:
            if chunks[j]["paper"] == q["paper_id"] and ans and ans in texts[j]:
                hit = 1
                break
        hits.append(hit)
    return hits

### Step 11 · Score every row

In [11]:
hits = {}
table = []
for name, chunks in chunk_sets.items():
    hits[name] = recall_at_5(chunks)
    table.append((name, len(chunks), sum(hits[name]) / len(questions), answer_findable(chunks)))

print(f"{'chunker':<20}{'chunks':>8}{'Recall@5':>11}{'answer findable':>18}")
for name, n, r, f in table:
    print(f"{name:<20}{n:>8}{r:>10.1%}{f:>17.1%}")

chunker               chunks   Recall@5   answer findable
fixed_512               4970     67.5%            94.2%
docling_512             5912     67.8%            94.3%
docling_512_label       5912     68.2%            94.3%


### Step 12 · Define the paired bootstrap CI

Every row answers the same 1,852 questions, so rows are compared question by question.

In [12]:
import random


def paired_ci(a, b, reps=2000, seed=0):
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(reps):
        s = 0
        for _ in range(n):
            k = rng.randrange(n)
            s += b[k] - a[k]
        diffs.append(s / n)
    diffs.sort()
    observed = sum(b[k] - a[k] for k in range(n)) / n
    return observed, diffs[int(0.025 * reps)], diffs[int(0.975 * reps)]

### Step 13 · Compare the rows

In [13]:
pairs = [("fixed_512", "docling_512"),
         ("docling_512", "docling_512_label"),
         ("fixed_512", "docling_512_label")]
for a, b in pairs:
    d, lo, hi = paired_ci(hits[a], hits[b])
    verdict = "difference" if lo > 0 or hi < 0 else "tie (CI crosses 0)"
    print(f"{b} - {a}: {d * 100:+.1f} pts  95% CI [{lo * 100:+.1f}, {hi * 100:+.1f}]  {verdict}")

docling_512 - fixed_512: +0.2 pts  95% CI [-1.0, +1.5]  tie (CI crosses 0)


docling_512_label - docling_512: +0.4 pts  95% CI [-0.3, +1.1]  tie (CI crosses 0)


docling_512_label - fixed_512: +0.6 pts  95% CI [-0.6, +1.9]  tie (CI crosses 0)


### Step 14 · Save the scores

In [14]:
results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
summary = {"n_questions": len(questions), "n_papers": len(paper_ids), "rows": []}
for name, n, r, f in table:
    summary["rows"].append({"chunker": name, "chunks": n, "recall_at_5": round(r, 4),
                            "answer_findable": round(f, 4)})
(results_dir / "recall_at_5_bm25.json").write_text(json.dumps(summary, indent=2))
print("saved results/recall_at_5_bm25.json")

saved results/recall_at_5_bm25.json


## What this shows

- On text questions with BM25, the three chunkers land within about one point of each
  other, and every paired CI crosses zero. The chunker is not what limits retrieval here.
- About 6% of answers are split across a boundary by every chunker. That is the
  ceiling no retriever can pass.
- Most of the misses are answers that sit whole in some chunk that BM25 did not rank
  in the top 5. That gap belongs to the retriever (hybrid search, reranking), not to
  chunking.

## Known gaps

- Text match only: 108 questions whose gold answer is not an exact span (`match` =
  `fuzzy` or `none`) count as misses for every row equally.
- BM25 only. The title label may matter more for a dense retriever; that is the next run.
- Table questions are not reported separately yet; that is where Docling's header
  repetition should show.
- The semantic chunker in `02-covidqa-intrinsic-metrics.ipynb` groups sentences with a
  16-dimension hash stand-in, not a real embedding model, so it is left out of this
  comparison.